**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**01 | Source and Reference Eligibility**

![Source and Reference Eligibility](../../images/01_source_and_reference_eligibility.svg)

**Objective:** Inventory original Camargo treadmill IMU and force-platform files, identify paired trials and audit their source channels and native timestamps before defining a scientific contact reference.

**Recorded source evidence:** The latest executed Notebook 01 and committed outputs cover **nine participants (AB06–AB14)**, **27 same-trial treadmill IMU/force pairs** and **135 original source files**. The source inventory includes 27 files each for IMU, force, `gcRight`, `gcLeft` and `conditions`; all 27 IMU/force pairs were decoded and have positive native-`Header` overlap. These are file-availability and timebase results, **not** independently validated force-contact references.

**Boundary:** Source-file pairing and common numeric time ranges do not by themselves establish force-axis direction, physical units, foot correspondence or event-detection performance.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
# This IPython-native checkout requires no imports outside Section 01.
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = "/content/human-movement-intelligence"
shell = get_ipython()
if shell is None:
    raise RuntimeError("Execute this notebook in Google Colab or IPython.")
shell.system(
    f'if [ ! -d "{repo_path}" ]; then git clone -q "{repo_url}" '
    f'"{repo_path}"; fi'
)

# ---------------------------- 2. Open Repository -----------------------------
shell.run_line_magic("cd", repo_path)


**01. Imports and Research Environment**

Initialise the shared Colab environment, the MATLAB table reader, scientific utilities and study output directory. Section 02 acquires only missing selected original source files, including matching gait-cycle and conditions streams when present. The planned nine-person source sample was completed in the latest recorded execution.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
from pathlib import Path
import subprocess
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 1

# ----------------------- 2. MATLAB Table Reader Setup ------------------------
import importlib.util
import sys

if importlib.util.find_spec("matio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mat-io==1.0.2"],
        check=True,
    )

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import os
import shutil
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 5. Study Research Utilities ------------------------
from camargo_source import (
    inventory_files, list_archives, paired_trials,
    read_sensor_table, validate_timebase, inspect_columns,
    locate_pilot_source_archive, download_mendeley_pilot_archive,
    PART1_V1_SUBJECT_ARCHIVES, extract_selected_trial_pairs_from_zip, 
    participant_pair_counts, extract_selected_auxiliary_sources_from_zip,
)
from gait_alignment import candidate_channels, preview_channels

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
source_root = Path(os.environ.get(
    "CAMARGO_DATA_ROOT", root / "data/raw/camargo"
))
output_dir = study_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Original source root:", source_root)
print("Scientific output folder:", output_dir)


**02. Prepare the Selected Camargo Source Cohort in Colab**

The scope remains **AB06–AB14, with three deterministically selected treadmill IMU/force pairs per person**. This stage fetches subject archives sequentially within one notebook execution; only newly downloaded temporary ZIPs are discarded. The total network transfer may approach 9 GB across nine original archives. An interrupted run can resume from the original files already extracted in the same runtime.

For the **same selected trials**, the original `gcRight`, `gcLeft` and `conditions` files were successfully extracted: **27 files of each type (81 supplementary files in total)**. Their verified filesystem availability is saved in `01_auxiliary_source_inventory.csv`; this does not verify the scientific meaning or independence of their event information. This does not change the predefined participant or trial-selection protocol.

All original files remain under `/content/human-movement-intelligence/data/raw/camargo/` inside the active Colab environment. These files are not committed to GitHub. **Run Notebook 01 and Notebook 02 in the same Colab runtime.** After a runtime reset, Notebook 01 must reconstruct the original selected files; published CSV summaries alone do not replace raw MAT recordings.


In [ ]:
# ================ 02. Prepare Resource-Bounded Camargo Cohort ================

# -------------------- 1. Cohort and Acquisition Settings ---------------------
TARGET_SUBJECTS = tuple(name.removesuffix(".zip")
                        for name in PART1_V1_SUBJECT_ARCHIVES)
MAX_TRIAL_PAIRS_PER_PERSON = 3
MAX_NEW_SUBJECTS_PER_RUN = len(TARGET_SUBJECTS)
AUTO_DOWNLOAD_PILOT = True
DELETE_NEW_ARCHIVE_AFTER_EXTRACTION = True
AUXILIARY_SENSORS = ("gcRight", "gcLeft", "conditions")
PILOT_ZIP_PATH = None  # Optional local AB06 source ZIP.

# -------------------- 2. Inspect Already Available Files ---------------------
source_root.mkdir(parents=True, exist_ok=True)
counts = participant_pair_counts(source_root, TARGET_SUBJECTS)
available = paired_trials(inventory_files(source_root))
available = available.loc[
    available.candidate_pair & available.condition.eq("treadmill") &
    available.subject_id.isin(TARGET_SUBJECTS)
].copy() if not available.empty else available
pending = []
for subject_id in TARGET_SUBJECTS:
    retained = available.loc[available.subject_id.eq(subject_id)]
    if len(retained) < MAX_TRIAL_PAIRS_PER_PERSON:
        pending.append(subject_id)
        continue
    selected = retained.sort_values(
        ["session_id", "trial_id"]
    ).head(MAX_TRIAL_PAIRS_PER_PERSON)
    missing_phase = any(
        not (source_root / Path(item.fp_path).parent.parent / sensor /
             Path(item.fp_path).name).is_file()
        for item in selected.itertuples(index=False)
        for sensor in ("gcRight", "gcLeft")
    )
    if missing_phase:
        pending.append(subject_id)
print("Existing original paired trials:", counts)
print("Participants requiring new source extraction:", pending)
print("Original data live in this Colab runtime; use the same runtime "
      "for the next notebook.")

# ---------------- 3. Prepare Archives Sequentially and Safely ----------------
for subject_id in pending[:MAX_NEW_SUBJECTS_PER_RUN]:
    print("Preparing participant:", subject_id)
    archive = locate_pilot_source_archive(
        source_root, subject_id=subject_id,
        explicit_archive=(PILOT_ZIP_PATH if subject_id == "AB06" else None),
    )
    downloaded = False
    if archive is None:
        if not AUTO_DOWNLOAD_PILOT:
            raise FileNotFoundError(
                f"Missing original {subject_id} ZIP. Supply a local source "
                "archive or enable automatic acquisition."
            )
        free_bytes = shutil.disk_usage(source_root).free
        if free_bytes < 2_200_000_000:
            raise OSError(
                f"Insufficient Colab disk space ({free_bytes/1e9:.2f} GB). "
                "Keep existing extracted MAT files; remove only disposable "
                "temporary downloads before continuing."
            )
        archive = download_mendeley_pilot_archive(
            source_root, filename=f"{subject_id}.zip",
            dataset_id="fcgm3chfff", version=1,
        )
        downloaded = True
    extraction_complete = False
    try:
        retained = available.loc[available.subject_id.eq(subject_id)]
        if len(retained) >= MAX_TRIAL_PAIRS_PER_PERSON:
            selected = retained.sort_values(
                ["session_id", "trial_id"]
            ).head(MAX_TRIAL_PAIRS_PER_PERSON)
        else:
            selected = extract_selected_trial_pairs_from_zip(
                archive, source_root, subject_id=subject_id,
                condition="treadmill",
                max_pairs=MAX_TRIAL_PAIRS_PER_PERSON,
            )
        supplementary = extract_selected_auxiliary_sources_from_zip(
            archive, source_root, selected, sensors=AUXILIARY_SENSORS,
        )
        print("Original auxiliary source status:",
              supplementary.source_status.value_counts().to_dict())
        extraction_complete = True
    finally:
        if (downloaded and extraction_complete
                and DELETE_NEW_ARCHIVE_AFTER_EXTRACTION):
            archive.unlink(missing_ok=True)
            print("Removed temporary ZIP downloaded after extraction.")

# ------------------ 4. Record Selected Source Availability -------------------
coverage = participant_pair_counts(source_root, TARGET_SUBJECTS)
current = paired_trials(inventory_files(source_root))
current = current.loc[
    current.candidate_pair & current.condition.eq("treadmill") &
    current.subject_id.isin(TARGET_SUBJECTS)
]
source_rows = []
for item in current.itertuples(index=False):
    for sensor in AUXILIARY_SENSORS:
        expected = (Path(item.fp_path).parent.parent / sensor /
                    Path(item.fp_path).name)
        found = (source_root / expected).is_file()
        source_rows.append({
            "subject_id": item.subject_id,
            "session_id": item.session_id, "condition": item.condition,
            "trial_id": item.trial_id, "sensor": sensor,
            "relative_path": expected.as_posix() if found else "",
            "source_status": (
                "available" if found else "not_available_in_runtime"
            ),
        })
auxiliary_availability = pd.DataFrame(source_rows, columns=[
    "subject_id", "session_id", "condition", "trial_id",
    "sensor", "relative_path", "source_status",
])
auxiliary_availability.to_csv(
    output_dir / "01_auxiliary_source_inventory.csv", index=False
)
print("Source pairs now available:", coverage)
if not auxiliary_availability.empty:
    print("Auxiliary-source availability:",
          auxiliary_availability.groupby(
              "sensor"
          ).source_status.value_counts().to_dict())
if any(n < MAX_TRIAL_PAIRS_PER_PERSON for n in coverage.values()):
    raise RuntimeError(
        "Source acquisition incomplete. Rerun this same Section 02 "
        "without discarding already extracted original MAT files."
    )
print("Selected original acquisition complete. Run Sections 03–07, "
      "then Notebook 02 in the SAME Colab runtime.")


**03. Inspect Original Archive and Sensor Inventory**

Inventory the original source files in the active Colab runtime, preserving subject, session, trial, sensor, source path and observed file size. The **current published inventory contains 135 original MAT files** for nine participants and 27 selected trials: **27 each** for `imu`, `fp`, `gcRight`, `gcLeft` and `conditions`. All 81 supplementary files are also reported as available in `01_auxiliary_source_inventory.csv`. The downloaded participant ZIPs were removed after extraction; an empty ZIP inventory is therefore expected. The original MAT files are **not stored in GitHub** and must be reacquired after a Colab runtime reset.


In [ ]:
# ======================= 03. Original Source Inventory =======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
if not source_root.exists():
    raise FileNotFoundError(
        f"No Camargo source found at {source_root}. "
        "Download/extract one participant (ABxx) from the official Georgia Tech "
        "Camargo dataset, or set CAMARGO_DATA_ROOT to the mounted data folder."
    )
archives = list_archives(source_root)
inventory = inventory_files(source_root)
display(archives.head(10), inventory.head(15))
print("Observed subjects:", inventory.subject_id.nunique())
print("Observed sensor records:", inventory.sensor.value_counts().to_dict())

# ------------------------- 2. Save Scientific Tables -------------------------
inventory.to_csv(output_dir / "01_trial_inventory.csv", index=False)
if inventory.empty:
    raise ValueError(
        "No extracted ABxx/.../imu or fp source records were discovered. "
        "ZIP files alone are not decoded; inspect archive structure and "
        "extract one participant before proceeding."
    )


**04. Identify Candidate Paired Trials**

Match files by their original participant, session, condition and trial names. The executed source audit identified **27 matched treadmill IMU/force pairs**, three for each participant AB06–AB14, with no missing selected original sensor files in the published inventory. The 81 additional source files correspond to those same 27 trial identities.

A pair is a *file-identity match* only. Timestamp compatibility, force-side mapping and valid event extraction are separate scientific checks.


In [ ]:
# ===================== 04. Identify Paired Sensor Trials =====================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
pairs = paired_trials(inventory)
display(pairs.head(15))
print("Candidate IMU/FP pairs:", int(pairs.candidate_pair.sum()))
pairs.to_csv(output_dir / "01_candidate_pairs.csv", index=False)
if not pairs.candidate_pair.any():
    raise ValueError(
        "No same-trial IMU/force pairs were detected. Inspect source paths "
        "and exact trial-name conventions before extending the adapter."
    )


**05. Audit Every Recorded Source Pair**

Read the original IMU and force-platform MATLAB tables **one pair at a time** to control peak memory. For every matched trial, inspect original channel names and `Header` vectors, confirm finite increasing timestamps, and record each stream's first/last time, median native step and observed intersection.

Two current evidence files are written: `01_observed_columns.csv` (original numeric sensor fields by participant/trial) and `01_trial_timebase_audit.csv` (per-trial read status and descriptive timebase statistics). Only the first matched pair is retained in memory for the example plot in Section 06.

**Scientific limitation:** An overlapping source `Header` interval is a necessary compatibility check, not independent proof of exact event alignment, Newton units, force polarity or anatomical side.


In [ ]:
# =============== 05. Audit Source Trials and Sensor Timebases ================

# -------------- 1. Review Every Available Matched Source Trial ---------------
observed_rows = []
timebase_rows = []
candidate_pairs = pairs.loc[pairs.candidate_pair].copy()
for item in candidate_pairs.itertuples(index=False):
    trial_meta = {
        "subject_id": item.subject_id,
        "session_id": item.session_id,
        "condition": item.condition,
        "trial_id": item.trial_id,
    }
    try:
        imu_path = source_root / item.imu_path
        fp_path = source_root / item.fp_path
        observed_imu = read_sensor_table(imu_path)
        observed_fp = read_sensor_table(fp_path)
        times_imu = validate_timebase(observed_imu, "IMU")
        times_fp = validate_timebase(observed_fp, "force")
        start = max(float(times_imu[0]), float(times_fp[0]))
        end = min(float(times_imu[-1]), float(times_fp[-1]))
        overlap = max(0.0, end - start)
        timebase_rows.append({
            **trial_meta, "status": "readable_with_time_overlap"
            if overlap > 0 else "no_native_time_overlap",
            "imu_start": float(times_imu[0]),
            "imu_end": float(times_imu[-1]),
            "force_start": float(times_fp[0]),
            "force_end": float(times_fp[-1]),
            "imu_median_step": float(np.median(np.diff(times_imu))),
            "force_median_step": float(np.median(np.diff(times_fp))),
            "native_overlap": overlap,
            "start_difference": float(times_imu[0] - times_fp[0]),
            "end_difference": float(times_imu[-1] - times_fp[-1]),
            "read_error": "",
        })
        for table, relative, modality in (
            (observed_imu, item.imu_path, "imu"),
            (observed_fp, item.fp_path, "fp"),
        ):
            fields = inspect_columns(table, relative).assign(
                modality=modality, **trial_meta
            )
            observed_rows.append(fields)
    except (OSError, ValueError, TypeError, KeyError) as error:
        timebase_rows.append({
            **trial_meta, "status": "source_read_or_timebase_error",
            "read_error": f"{type(error).__name__}: {error}",
        })
timebase_audit = pd.DataFrame(timebase_rows)
if observed_rows:
    column_audit = pd.concat(observed_rows, ignore_index=True)
else:
    column_audit = pd.DataFrame()


# -------------- 2. Save Trial-Level Audits and Inspect Coverage --------------
timebase_audit.to_csv(
    output_dir / "01_trial_timebase_audit.csv", index=False
)
column_audit.to_csv(
    output_dir / "01_observed_columns.csv", index=False
)
display(timebase_audit, column_audit.head(30))
print("Reviewed matched source pairs:", len(candidate_pairs))
print("Readable source pairs with native-time overlap:", int(
    (timebase_audit.status == "readable_with_time_overlap").sum()
))
print("Source readings do not independently validate a common clock, "
      "force orientation, physical units or contact events.")
if column_audit.empty:
    raise ValueError(
        "No original IMU/force tables were decoded. Review the saved "
        "01_trial_timebase_audit.csv before continuing."
    )


# ------------ 3. Retain One Recorded Trial for a Descriptive Plot ------------
sample = candidate_pairs.iloc[0]
imu_file = source_root / sample.imu_path
fp_file = source_root / sample.fp_path
imu_table = read_sensor_table(imu_file)
fp_table = read_sensor_table(fp_file)
imu_time = validate_timebase(imu_table, "IMU")
fp_time = validate_timebase(fp_table, "force")
print("Plotted example:", sample.subject_id, sample.trial_id)
print("Native IMU step:", float(np.median(np.diff(imu_time))))
print("Native force step:", float(np.median(np.diff(fp_time))))
print("IMU channel candidates:", candidate_channels(imu_table, "imu"))
print("Force source components:", candidate_channels(fp_table, "fp"))


**06. Visualise Recorded IMU and Treadmill Source Channels**

The two-panel recorded plot illustrates the first matched trial only, using original `foot_Gyro_Y`, `shank_Gyro_Y`, `Treadmill_R_vy` and `Treadmill_L_vy` fields when available. The source `Header` is used separately for each panel; the plot does not artificially synchronise streams.

The study-wide source audit is the per-trial CSV written in Section 05. The visible component directions, force sign, physical units and side correspondence remain subject to the independent sensor-contract assessment in Notebook 02. No IC or TC ground truth is declared in this figure.


In [ ]:
# ======================= 06. Visualise Source Signals ========================

# ------------------ 1. Choose Observed Source Channel Names ------------------
imu_channels = preview_channels(imu_table, "imu")
force_channels = preview_channels(fp_table, "fp")
if not imu_channels or not force_channels:
    raise ValueError("Insufficient numeric sensor columns for preview.")
print("IMU gyroscope channels:", imu_channels)
print("Treadmill force-platform components:", force_channels)
print("These plotted channels are not yet validated contact references.")

# --------------- 2. Plot Short Recorded Native-Time Intervals ----------------
fig, axes = plt.subplots(
    2, 1, figsize=(11, 6.4), sharex=False, layout="constrained"
)
panels = (
    (axes[0], imu_table, imu_time, imu_channels,
     "Observed IMU gyroscope", ("#236CDE", "#168C5B")),
    (axes[1], fp_table, fp_time, force_channels,
     "Observed treadmill force-platform components",
     ("#E87929", "#236CDE")),
)
for ax, table, time, columns, title, colours in panels:
    native_time = np.asarray(time, dtype=float)
    view = native_time <= native_time[0] + 5.0
    if np.count_nonzero(view) < 3:
        raise ValueError("Too few native-time samples for signal preview.")
    for column, colour in zip(columns[:2], colours):
        signal = pd.to_numeric(table.loc[view, column], errors="coerce")
        ax.plot(native_time[view], signal, color=colour,
                linewidth=1.0, label=column)
    ax.set_title(title, loc="left")
    ax.set_ylabel("Channel value (source units)")
    ax.set_xlabel("Stored Header (source time units)")
    ax.legend(loc="upper right", ncol=2, frameon=False)
    ax.grid(alpha=0.2)
fig.suptitle(
    f"{sample.subject_id} / {sample.trial_id}: recorded sensor signals"
)

# --------------------- 3. Export Observed Signal Preview ---------------------
save_figure(fig, output_dir / "01_sensor_source_preview.png")
plt.show()


**07. Research Summary and Output Download**

Summarise the **executed nine-person source sample**: AB06–AB14, 27 matched treadmill IMU/force pairs, **135 extracted original files** across five source types, **81 matching supplementary files** and 27 readable IMU/force pairs with overlapping native `Header` intervals. The selected source acquisition and source-timebase audit are complete, but scientific contact-reference eligibility remains pending. Do not confuse these inventory statistics with the number of participants eligible for a force-referenced experiment.

The executed output stage packaged **seven `01_` files**: candidate pairs, trial inventory, auxiliary-source inventory, observed numeric fields, trial-level timebase audit, eligibility summary and the measured IMU/force preview. The existing two-panel figure displays real original signals for AB06/`treadmill_01_01`; no redrawing or numerical alteration is required. No raw sensor recording is included.

**Next scientific task:** Notebook 02 must examine the selected gyroscope axis and right/left force component, physical units, force sign and independently validated common timestamps. Neither a completed nine-person download nor an overlap in `Header` values automatically authorises force-contact labels.


In [ ]:
# ================= 07. Research Summary and Output Download ==================

# ---------------- 1. Summarise Current Source-Audit Coverage -----------------
coverage = participant_pair_counts(source_root, TARGET_SUBJECTS)
ready = sum(n >= MAX_TRIAL_PAIRS_PER_PERSON for n in coverage.values())
readable = int((timebase_audit.status == "readable_with_time_overlap").sum())
summary = pd.DataFrame([
    {"item": "source_participants_target", "value": len(TARGET_SUBJECTS)},
    {"item": "source_participants_observed",
     "value": int(inventory.subject_id.nunique())},
    {"item": "source_participants_with_selected_pairs",
     "value": int(ready)},
    {"item": "source_acquisition_complete",
     "value": bool(ready == len(TARGET_SUBJECTS))},
    {"item": "candidate_imu_force_pairs",
     "value": int(pairs.candidate_pair.sum())},
    {"item": "readable_pairs_with_native_time_overlap",
     "value": readable},
    {"item": "validated_reference_pairs",
     "value": "not yet independently established"},
    {"item": "force_contact_event_results",
     "value": "not generated in source inspection"},
])
display(summary)


# --------------------- 2. Save Scientific Source Summary ---------------------
summary.to_csv(output_dir / "01_source_eligibility_summary.csv", index=False)

# ---------------------- 3. Package Notebook 01 Evidence ----------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
